In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the list $[\texttt{start}, \cdots, \texttt{stop}]$.

In [ ]:
let range (start : int) (stop : int) : int list =
  List.init (max 0 (stop - start + 1)) (fun i -> start + i)

In [ ]:
range 3 5

In [ ]:
import_notebook "02-Backtracking-Constraint-Solver.ipynb"

# Saving the Infidels

In this notebook we want to solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossings of the river.  We use the following variables:
* $\texttt{m}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the
  $i^{\textrm{th}}$ crossing.
* $\texttt{c}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the
  $i^{\textrm{th}}$ crossing.
* $\texttt{b}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the
  $i^{\textrm{th}}$ crossing.

The TypeScript version uses upper case letters for these variables.  In <samp>OCaml</samp>, variables have to start with a lower case letter.

## Auxiliary Functions

The following functions generate the *formulas* required by the constraint solver.  A formula is a string containing an <samp>OCaml</samp> expression of type `bool`.

The invariant is given by the following formula:
$$M = 0 \vee M = 3 \vee M = C$$
The function `invariant` takes three arguments:
- `m` is the variable for the number of missionaries on the western shore,
- `c` is the variable for the number of infidels on the western shore,
- `b` is the variable for the number of boats on the western shore.  It is not needed for the invariant.

It returns the invariant as a formula.  The invariant states that the missionaries on the western shore are not outnumbered by the infidels.  As the number of missionaries on the eastern shore is $3 - M$, the same holds for the eastern shore.

In [ ]:
let invariant (m : string) (c : string) (b : string) : string =
  ignore b;
  Printf.sprintf "%s = 0 || %s = 3 || %s = %s" m m m c

In [ ]:
invariant "m0" "c0" "b0"

The formula for the transition has three components:
* The boat changes sides with every transition:
  $$B_\beta = 1 - B_\alpha$$
* If the boat goes from the west to the east, there are two conditions:
  + The number of people on the boat must be at least 1 and at most two.
    $$1 \leq (M_\alpha - M_\beta) + (C_\alpha - C_\beta) \leq 2$$
  + The number of both missionaries and infidels on the western side must not increase.
    $$M_\beta \leq M_\alpha \wedge C_\beta \leq C_\alpha$$
* If the boat goes from the east to the west, there are two conditions:
  + The number of people on the boat must be at least 1 and at most two.
    $$ 1 \leq (M_\beta - M_\alpha) + (C_\beta - C_\alpha) \;\wedge\;
              (M_\beta - M_\alpha) + (C_\beta - C_\alpha)\leq 2 $$
  + The number of both missionaries and infidels on the western side must not decrease.
    $$M_\beta \geq M_\alpha \wedge C_\beta \geq C_\alpha$$

The function `transition` takes six arguments:
- `m_a`, `c_a`, and `b_a` are the variables describing the state before the crossing,
- `m_b`, `c_b`, and `b_b` are the variables describing the state after the crossing.

It returns the formula describing the transition.  The TypeScript version uses the conditional operator `? :`.  In <samp>OCaml</samp>, the expression `if b then e1 else e2` can be used inside of other expressions.

In [ ]:
let transition (m_a : string) (c_a : string) (b_a : string)
               (m_b : string) (c_b : string) (b_b : string) : string =
  let west_to_east =
    [Printf.sprintf "1 <= (%s - %s) + (%s - %s)" m_a m_b c_a c_b;
     Printf.sprintf "(%s - %s) + (%s - %s) <= 2" m_a m_b c_a c_b;
     Printf.sprintf "%s <= %s && %s <= %s" m_b m_a c_b c_a] in
  let east_to_west =
    [Printf.sprintf "1 <= (%s - %s) + (%s - %s)" m_b m_a c_b c_a;
     Printf.sprintf "(%s - %s) + (%s - %s) <= 2" m_b m_a c_b c_a;
     Printf.sprintf "%s >= %s && %s >= %s" m_b m_a c_b c_a] in
  Printf.sprintf "%s = 1 - %s && (if %s = 1 then %s else %s)"
    b_b b_a b_a (String.concat " && " west_to_east) (String.concat " && " east_to_west)

In [ ]:
transition "m0" "c0" "b0" "m1" "c1" "b1"

The function `missionaries_csp` takes one argument:
- `n` is the number of crossings.

It creates a CSP that tries to solve the problem with `n` crossings.

In [ ]:
let missionaries_csp (n : int) : csp =
  let var x i = Printf.sprintf "%s%d" x i in
  let vars    = List.concat_map (fun i -> [var "m" i; var "c" i; var "b" i]) (range 0 n) in
  let values  = range 0 3 in
  let constrs =
    ["m0 = 3 && c0 = 3 && b0 = 1";                                      (* Start *)
     Printf.sprintf "%s = 0 && %s = 0 && %s = 0" (var "m" n) (var "c" n) (var "b" n)]  (* Goal *)
    @ List.map (fun i -> invariant (var "m" i) (var "c" i) (var "b" i)) (range 0 (n - 1))
    @ List.map (fun i -> transition (var "m" i) (var "c" i) (var "b" i)
                                    (var "m" (i + 1)) (var "c" (i + 1)) (var "b" (i + 1)))
               (range 0 (n - 1))
  in
  { variables = vars; values = values; constraints = constrs }

In [ ]:
missionaries_csp 3

The function `find_solution` takes the argument `()`.  It computes a solution to the problem of saving the infidels by iteratively increasing the number of crossings `n`.  As the boat has to end on the eastern shore, the number of crossings is odd.  The result is the pair `(n, solution)`.

In [ ]:
let find_solution () : int * assignment =
  let rec try_crossings n =
    Printf.printf "Checking n = %d...\n" n;
    match solve (missionaries_csp n) with
    | Some solution -> (n, solution)
    | None          -> try_crossings (n + 2)
  in
  try_crossings 1

Finding the solution takes less than a second.

In [ ]:
let (n, solution) = time "find_solution" find_solution

## Visualizing the Result

The function `repeat` takes two arguments:
- `s` is a string,
- `k` is a natural number.

It returns the string that consists of `k` copies of `s`.

The function `show_solution` takes two arguments:
- `solution` is a solution of the CSP,
- `n` is the number of crossings.

It prints the states of the solution and the passengers of the boat for every crossing.  As the emojis are twice as wide as a blank, the gap between the two shores consists of $28 - 2 \cdot (M + C)$ blanks.

In [ ]:
let repeat (s : string) (k : int) : string =
  String.concat "" (List.init k (fun _ -> s))

let show_solution (solution : assignment) (n : int) : unit =
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) solution in
  for i = 0 to n do
    let m = get "m" i and c = get "c" i and b = get "b" i in
    let left_side  = repeat "😇" m ^ repeat "🥷" c in
    let right_side = repeat "😇" (3 - m) ^ repeat "🥷" (3 - c) in
    let gap        = String.make (28 - (m + c) * 2) ' ' in
    print_endline (left_side ^ gap ^ right_side);
    if i < n then begin
      let next_m = get "m" (i + 1) and next_c = get "c" (i + 1) in
      if b = 1 then
        Printf.printf "%s>>> %s%s >>>\n" (String.make 10 ' ') (repeat "😇" (m - next_m)) (repeat "🥷" (c - next_c))
      else
        Printf.printf "%s<<< %s%s <<<\n" (String.make 10 ' ') (repeat "😇" (next_m - m)) (repeat "🥷" (next_c - c))
    end
  done

In [ ]:
show_solution solution n

The function `show_animated_solution` takes two arguments:
- `solution` is a solution of the CSP,
- `n` is the number of crossings.

It displays an animation of the solution.  The animation is written in HTML and JavaScript.  With the buttons, the boat can be moved from one state to the next.

**Implementation:** The HTML code is stored in the quoted string `template`.  A quoted string `{|...|}` can contain line breaks and quotes.  The template contains the placeholders `__UID__`, `__N__`, and `__STATES__`:
- `__UID__` is replaced by a random identifier, so that the elements of different animations in the same notebook have different names,
- `__N__` is replaced by the number of crossings,
- `__STATES__` is replaced by the list of states in the JSON format, which can be read by JavaScript.

The function `replace_all s pattern by` replaces every occurrence of the string `pattern` in `s` by the string `by`.

In [ ]:
#require "jupyter.notebook";;

In [ ]:
let replace_all (s : string) (pattern : string) (by : string) : string =
  let n = String.length s and k = String.length pattern in
  let buffer = Buffer.create n in
  let rec loop i =
    if i > n - k then
      Buffer.add_string buffer (String.sub s i (n - i))
    else if String.sub s i k = pattern then begin
      Buffer.add_string buffer by;
      loop (i + k)
    end else begin
      Buffer.add_char buffer s.[i];
      loop (i + 1)
    end
  in
  loop 0;
  Buffer.contents buffer

let template = {|
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-__UID__" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-__UID__" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-__UID__" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-__UID__" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of __N__</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-m-__UID__" x="75" y="100" font-size="28" text-anchor="middle">😇😇😇</text>
            <text id="west-c-__UID__" x="75" y="150" font-size="28" text-anchor="middle">🥷🥷🥷</text>
            
            <text id="east-m-__UID__" x="525" y="100" font-size="28" text-anchor="middle"></text>
            <text id="east-c-__UID__" x="525" y="150" font-size="28" text-anchor="middle"></text>

            <g id="boat-__UID__" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-__UID__" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = __STATES__;
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-__UID__');
        const westM = document.getElementById('west-m-__UID__');
        const westC = document.getElementById('west-c-__UID__');
        const eastM = document.getElementById('east-m-__UID__');
        const eastC = document.getElementById('east-c-__UID__');
        const boatP = document.getElementById('boat-p-__UID__');
        const stepLabel = document.getElementById('label-step-__UID__');
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            // DIESEL ENGINE SIMULATION:
            // A sawtooth wave at ~12 Hz creates a rapid "clicking/knocking" 
            // sound that mimics the slow chug of a diesel engine.
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            
            // Low-pass filter to muffle the harsh highs and keep it sounding bassy/heavy
            filter.type = 'lowpass';
            filter.frequency.value = 150;
            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            // Using a slightly higher gain (0.6) because extreme low frequencies are perceived as quieter
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4); // Slower fade out for realism
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            westM.textContent = '😇'.repeat(s.M);
            westC.textContent = '🥷'.repeat(s.C);
            eastM.textContent = '😇'.repeat(3 - s.M);
            eastC.textContent = '🥷'.repeat(3 - s.C);
            boatP.textContent = '';
            
            boat.style.transform = s.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of __N__';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const mPass = Math.abs(s0.M - s1.M);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.B === 1) { 
                westM.textContent = '😇'.repeat(s0.M - mPass);
                westC.textContent = '🥷'.repeat(s0.C - cPass);
            } else { 
                eastM.textContent = '😇'.repeat((3 - s0.M) - mPass);
                eastC.textContent = '🥷'.repeat((3 - s0.C) - cPass);
            }
            boatP.textContent = '😇'.repeat(mPass) + '🥷'.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-__UID__').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-__UID__').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-__UID__').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    |}

let show_animated_solution (solution : assignment) (n : int) : unit =
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) solution in
  let states =
    "[" ^ String.concat ", "
            (List.map (fun i -> Printf.sprintf "{\"M\": %d, \"C\": %d, \"B\": %d}" (get "m" i) (get "c" i) (get "b" i))
                      (range 0 n)) ^ "]"
  in
  let uid  = Printf.sprintf "%07x" (Random.bits ()) in
  let html = replace_all (replace_all (replace_all template "__UID__" uid) "__N__" (string_of_int n))
                         "__STATES__" states in
  ignore (Jupyter_notebook.display "text/html" html)

In [ ]:
show_animated_solution solution n